---
title: word2vec from Scratch
description: |
  Week 1 of the residency. How to turn free text into numbers a model can use, starting from the feature engineering problem that makes you need it.
author: Rosh Beed
date: '2026-06-08'
categories:
  - nlp
  - embeddings
  - week-1
jupyter: python3
---


Week 1 didn't start with word2vec. It started with a boring problem. word2vec was
where we ended up when the boring problem stopped working.

The problem: predict someone's income from what you know about them.

![](figures/slide-03.png){fig-alt="A person, a question mark, a small neural network, and an output of £40,000. Underneath, the words: convert the user's key characteristics into meaningful numbers the model can process."}

A neural network takes numbers in and gives a number out. The job is the caption
under that diagram. Turn what you know about a person into numbers.

Some of it's easy.

![](figures/slide-04.png){fig-alt="Gender: yes, 0 is male and 1 is female. Age: yes, a nice integer between 0 and 100. Height: yes, use centimetres. Weight: yes, kilograms. Education: hmm. City: oops. Job: oops."}

Gender is a flag. Age is already a number. Height in centimetres. Weight in
kilograms. Four features, no thought required.

Then education. The easy run stops.

## Encoding Education

![](figures/slide-05.png){fig-alt="Four encodings of education. Ordinal: none 0 through PhD 4, \"treats PhD as 4 times none\". Years of schooling: none 0 through PhD 21, \"requires some domain knowledge\". One-hot: a five-dimensional indicator, \"no assumptions, use more neurons\". Learned embeddings: a five-number vector per level, learned during training."}

There are four options on that slide.

**Ordinal.** None is 0, high school 1, bachelor 2, master 3, PhD 4. This encodes the
ordering. It also tells the model that a PhD is four times a high school diploma.
Nobody believes that.

**Years of schooling.** 0, 12, 15, 17, 21. Better, because the spacing means
something. But you had to know the answer. You supplied the domain knowledge, not
the model.

**One-hot.** Five slots, one of them set to 1. It makes no claims about ordering or
distance. The cost is a slot per category. It also makes every pair equally far
apart, so a master's and a PhD are as different as a PhD and no schooling.

**Learned embeddings.** Give each level a short list of numbers. Start them random.
Let training move them. No assumptions, no ordering imposed. The model decides what
the numbers mean.

The fourth option is what this week is about. At this scale it looks like overkill.
There are five categories. One-hot would do.

## Encoding City

![](figures/slide-06.png){fig-alt="The same list, with education now filled in as 3 for master, and city and job still marked \"oops\"."}

City has the same problem as education. No ordering, and a thousand categories
instead of five. One-hot is now a thousand slots of mostly zeros.

There is a way out.

![](figures/slide-07.png){fig-alt="City decomposed into population size 1M, average age 31, average income 22k, crime rate 4, all feeding the network."}

Break the category into features that describe it. A city is its population, its
average age, its average income, its crime rate. Four numbers instead of a
thousand. Each one means something. Two similar cities end up with similar numbers.

This is good feature engineering. A lot of real models are built this way. It also
has a ceiling, and the next feature is above it.

## Free Text

![](figures/slide-09.png){fig-alt="A free-text list of hobbies, interests and life story, several sentences long, with a red \"oops\" and a column of X marks. Underneath, in red: how can we produce meaningful numbers from free text?"}

Now the form has a box that says *tell us about yourself*. Someone has written a
paragraph.

There is no decomposition to reach for. Cities have populations. Paragraphs do not
have an agreed set of four numbers. The categories are unbounded, because anyone
can write anything. Every trick from the last three slides is gone.

The red question on that slide is the subject of week 1:

> How can we produce meaningful numbers from free text?

## The Distributional Hypothesis

What is the meaning of *bardiwac*?

> He handed her her glass of **bardiwac**.
>
> Beef dishes are made to complement the **bardiwacs**.
>
> Nigel staggered to his feet, face flushed from too much **bardiwac**.
>
> I dined off bread and cheese and this excellent **bardiwac**.

You have never seen that word before. After four sentences you know it is a drink.
Probably a red wine. Probably alcoholic.

Nobody defined it for you. You got it from the company it keeps.

This is the distributional hypothesis. **A word can be described by the words that
appear around it.** If that holds, you can compute a word's meaning from a large
pile of ordinary text. Nobody has to label anything.

The learned-embedding option is still the plan. What changed is that there is now a
way to train it without labels.

## The word2vec Recipe

![](figures/slide-13.png){fig-alt="word2vec recipe in three steps. One: treat the entire corpus as one long string, slide a window of five tokens across it, mask the middle token and use the remaining four to predict it. Two: because any given context can sensibly be completed by several different words, don't force the model to output one correct token; teach it to return a probability distribution over the whole vocabulary. Three: CBOW."}

Three steps. The second one is the one people skip.

* Slide a window over the text
* Hide the middle word
* Predict it from its neighbours

The slide draws the window five tokens wide, so four neighbours predict the
middle one. Width is a dial: the run below takes five words *either* side, which
is what Mikolov's paper uses and what the deployed service is configured with.

That gives you an enormous number of training examples from raw text, with no
annotation.

Step two is the important bit. "For dinner we served dark red ___ with steak" does
not have one right answer. It could be *bardiwac*, *wine*, *merlot* or *malbec*. So
the model is not trained to output one word. It is trained to output a **score for
every word in the vocabulary**.

That is the part that makes it work. Words that fit the same gaps end up with
similar scores, and similar scores pull their vectors together.

There are two ways to arrange that prediction.

![](figures/slide-14.png){fig-alt="Continuous bag of words: \"Hello my ? is Bes\" with the four context words going through a tokenizer, an embedding layer, averaged into one vector, and projected to predict \"name\"."}

**CBOW** takes the context and predicts the middle word. Four words go in. Each is
looked up in a table of vectors. The four vectors are averaged into one. That one
vector is used to score every word in the vocabulary.

![](figures/slide-15.png){fig-alt="Skip-gram: the word \"name\" going through a tokenizer and embedding, predicting \"Hello\", \"my\", \"is\", \"Bes\", with the negative-sampling loss formula and reference plots of the sigmoid and logarithm."}

**Skip-gram** runs it the other way. One word in, predict each of its neighbours.

I trained both.

One thing to notice: that sentence, *Hello my name is Bes*. It comes back in week 3
going through a transformer, in week 5 being transcribed by Whisper, and in week 6
being scored by a reward model. It is the course's running example. By the end you
have seen the same five words pass through six architectures.

## Negative Sampling

Step two is expensive.

To turn raw scores into probabilities you need a **softmax**. Exponentiate each
score, then divide by the total so they add up to one. The dividing is the problem.
The total is over the whole vocabulary. Here that is 71,290 words, after dropping
everything seen fewer than five times.

So every training step touches 71,290 words to learn one thing.

Negative sampling changes the question. Instead of asking *which of these 71,290
words goes here*, ask *is this pair real, or did I make it up*. Once for the true
neighbour, then five more times for words picked at random. Six comparisons instead
of 71,290.

Where the random words come from matters:

* Pick uniformly and nearly every one is a rare word the model already scores near
  zero. It learns nothing.
* Pick by raw frequency and nearly all of them are `the`.
* Mikolov's papers raise the frequencies to the power 3/4, which sits between the
  two.

## Training It

The rest of this page trains that model. It is small enough to finish while the
page builds: two million characters of text, a fiftieth of the real run, 64 numbers
per word.

First the corpus itself, cut into words and counted. A word that turns up two or
three times in the whole text gives the model nothing to average over, so anything
appearing fewer than ten times is dropped before training.

In [ ]:
#| warning: false

# Setup, inlined rather than imported so this notebook runs on its own.
# Keep it folded; nothing below it depends on anything outside this file.

import matplotlib.pyplot as plt

# --- chart styling -------------------------------------------------------
# Categorical slots of a CVD-validated palette: blue, orange, aqua, purple.
COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#8a63d2"]
MUTED, GRID, AXIS = "#5b6570", "#e6e6e3", "#d5d5d1"


def style_axes(ax, xlabel=None, ylabel=None, grid="y"):
    """Strip an axes back to the ink that carries information."""
    if xlabel:
        ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    if grid:
        ax.grid(axis=grid, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    return ax


def figure(width=7.0, height=4.2, **kw):
    fig, ax = plt.subplots(figsize=(width, height), **kw)
    return fig, ax


import collections

import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

# Every build re-executes this page, so a result that shifts between runs would let
# the prose and the output disagree. Torch's multithreaded CPU reductions add floats
# in whatever order the threads finish in; over a training loop that compounds into a
# different model. One thread makes the run reproducible.
torch.set_num_threads(1)

REVISION = "9113ea48905b4f7178b333919ed5ec1a474561d7"
path = hf_hub_download("roshbeed/ai-residency-blog-data", "text8/text8-2m.txt",
                       repo_type="dataset", revision=REVISION)

words = open(path).read().split()
counts = collections.Counter(words)
vocab = [w for w, c in counts.most_common() if c >= 10]
stoi = {w: i for i, w in enumerate(vocab)}
ids = np.array([stoi[w] for w in words if w in stoi], dtype=np.int64)

print(f"{len(words):,} tokens, {len(counts):,} distinct, {len(vocab):,} kept")


A third of a million words, of which only a few thousand distinct ones clear that
bar. Those are the words the model will learn a vector for.

One more step before training. `the` appears about 21,000 times in that text, one
word in sixteen, and it sits next to everything, so it tells you nothing about its
neighbours. Common words get thrown
away, and the more common a word is the more often it goes.

In [ ]:
frequency = np.bincount(ids, minlength=len(vocab)).astype(np.float64)
frequency /= frequency.sum()

t = 1e-3
keep_probability = np.minimum(1.0, np.sqrt(t / frequency))
rng = np.random.default_rng(0)
kept = ids[rng.random(len(ids)) < keep_probability[ids]]

print(f"'the' survives with probability {keep_probability[stoi['the']]:.3f}, "
      f"'philosophy' with {keep_probability[stoi['philosophy']]:.3f}")

WINDOW = 5  # words either side of the centre, so 2 * WINDOW pairs per word
centres, contexts = [], []
for offset in range(1, WINDOW + 1):
    centres.append(kept[offset:]);  contexts.append(kept[:-offset])
    centres.append(kept[:-offset]); contexts.append(kept[offset:])

centre = torch.from_numpy(np.concatenate(centres))
context = torch.from_numpy(np.concatenate(contexts))
noise = torch.from_numpy(frequency ** 0.75 / (frequency ** 0.75).sum())
print(f"{len(centre):,} training pairs")

So `the` survives about one window in nine and `philosophy` survives every time.
Pairing each surviving word with the five words either side of it gives ten pairs
per word, and those pairs are what the model trains on.

There are two tables of vectors. One holds a word's vector when it's the centre of
a window. The other holds it when it's somebody's neighbour. Only the first is kept
at the end. The second exists to give the first something to be scored against.

The loss below pushes the true pair up and five invented pairs down.

In [ ]:
DIMENSIONS, K = 64, 5
generator = torch.Generator().manual_seed(0)

centre_vectors = (torch.randn(len(vocab), DIMENSIONS, generator=generator) * 0.01).requires_grad_()
# Small random, not zeros. Zeros force every dot product to 0, which makes the
# baseline below exactly (1 + K) * ln 2 by construction rather than by measurement.
context_vectors = (torch.randn(len(vocab), DIMENSIONS, generator=generator) * 0.01).requires_grad_()
optimiser = torch.optim.Adam([centre_vectors, context_vectors], lr=2e-3)

def loss_on(centre_batch, context_batch):
    """One real pair scored up, K invented pairs scored down."""
    v = centre_vectors[centre_batch]
    real = F.logsigmoid((v * context_vectors[context_batch]).sum(-1))
    fake_ids = torch.multinomial(noise, len(centre_batch) * K,
                                 replacement=True, generator=generator)
    fake = context_vectors[fake_ids.view(len(centre_batch), K)]
    invented = F.logsigmoid(-(fake @ v.unsqueeze(-1)).squeeze(-1)).sum(-1)
    return -(real + invented).mean()

with torch.no_grad():
    baseline = loss_on(centre[:8192], context[:8192]).item()
print(f"loss before training: {baseline:.3f}, and (1 + {K}) * ln 2 = {(1 + K) * np.log(2):.3f}")

Those two numbers matching is a useful check. A model that knows nothing is
guessing on six yes/no questions, and each costs ln 2.

It is worth being precise about what that catches, because a check you trust too
far is worse than none. It pins down the *arithmetic around* the loss: that there
really are `K` negatives and not three, that they are summed while the batch is
averaged, that nothing is double-counted. It does not catch a sign error. At
initialisation the vectors are near zero, so every dot product is near zero, and
sigmoid is a half either way. Flip the minus in front of the invented pairs and this
line still prints 4.159.

So the baseline is worth printing next to the trained loss, and worth not treating
as proof the loss is right.

Now run it. Twenty-five passes over those pairs, and then, as a first look at what
came out, the six nearest words to a handful of test words.

In [ ]:
EPOCHS, BATCH = 25, 8192
history = []

for epoch in range(EPOCHS):
    order = torch.randperm(len(centre), generator=generator)
    total = steps = 0
    for i in range(0, len(order) - BATCH, BATCH):
        batch = order[i:i + BATCH]
        loss = loss_on(centre[batch], context[batch])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        total += loss.item()
        steps += 1
    history.append(total / steps)

print(f"loss {baseline:.3f} -> {history[-1]:.3f} over {EPOCHS} epochs")


Down from the six-coin baseline, which is the only thing that number means on its
own. What it bought is easier to see in the vectors. Here are the six nearest words
to a handful of test words, by cosine similarity:

In [ ]:
embeddings = F.normalize(centre_vectors.detach(), dim=1)

def nearest(word, k=6):
    similarity = embeddings @ embeddings[stoi[word]]
    similarity[stoi[word]] = -1
    return [vocab[i] for i in similarity.topk(k).indices.tolist()]

for word in ("king", "france", "computer", "three", "war", "music"):
    print(f"{word:10} {', '.join(nearest(word))}")

Nothing labelled any of that. The lists come out of counting which words turn up
near which.

Sixty-four numbers per word is more than can be drawn, so the last step is to flatten
them onto a plane. Take three groups of words that have nothing in common except
their category, find the two directions that separate the group of them most, and
plot only those.

In [ ]:
#| label: fig-space
#| fig-cap: Thirty words from three categories, projected onto the two directions that spread them furthest apart. Nothing in training was told these categories exist.
#| fig-alt: A scatter plot of thirty labelled words in three colours. The number words sit in one corner, packed so tightly their labels overlap; the country names spread across the opposite side and the time words sit below, with no overlap between the three groups.

import matplotlib.pyplot as plt

GROUPS = {
    "numbers": ["one", "two", "three", "four", "five", "six", "seven", "eight",
                "nine", "ten", "zero"],
    "countries": ["france", "germany", "italy", "spain", "russia", "china", "japan",
                  "england", "greece", "egypt", "india"],
    "time": ["january", "february", "march", "april", "year", "century", "day", "month"],
}
present = {name: [w for w in words if w in stoi] for name, words in GROUPS.items()}
picked = [w for words in present.values() for w in words]

# Principal components of just these words: the two directions along which this
# particular set of thirty spreads out most. Not a general map of the space.
M = embeddings[[stoi[w] for w in picked]].numpy()
M = M - M.mean(0)
xy = M @ np.linalg.svd(M, full_matrices=False)[2][:2].T

fig, ax = figure(width=7.0, height=5.2)
start = 0
for (name, words), colour in zip(present.items(), COLOURS):
    points = xy[start:start + len(words)]
    ax.scatter(points[:, 0], points[:, 1], s=40, color=colour, zorder=3, label=name)
    for word, (x, y) in zip(words, points):
        ax.annotate(word, (x, y), xytext=(5, 3), textcoords="offset points",
                    fontsize=8.5, color=colour)
    start += len(words)

ax.legend(frameon=False, fontsize=9, labelcolor=MUTED, loc="best")
style_axes(ax, "First principal direction", "Second", grid=None)
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()


Three groups in three regions. That is the answer to the red question: every word
now has 64 numbers attached, and the numbers put related words near each other.
Nothing was labelled to make it happen.

The small numbers are worth a second look. They are packed so tightly their labels
sit on top of each other, because `three` and `four` and `seven` turn up in almost
identical company. `ten` has drifted away from them, and `zero` and `two` sit out
at the edge of the group. The distances are carrying information, not just the
clusters.

To turn a paragraph into numbers, look up each word and average. Crude, and enough
to close the loop we opened with:

![](figures/slide-10.png){fig-alt="The same person and network diagram, now with a Description field carrying the free-text summary into the model alongside gender, age, height, weight and education, producing £40,000."}

The free-text box goes in. Every feature on that form is now a number.

The full run uses fifty times this much text. It scores at or above the published
word2vec numbers on the same data:

| benchmark | untrained | skip-gram | CBOW | published |
|---|---|---|---|---|
| WordSim-353 | 0.028 | 0.726 | **0.730** | 0.68 |
| SimLex-999 | 0.008 | **0.297** | 0.290 | 0.30 |
| Google analogies | 0.000 | **0.464** | 0.408 | 0.38 |

The untrained column is not decoration. A checkpoint in this project once reported
a loss of 10.02. That reads like a trained model. An untrained one reports 11.09.

## Checking the Vectors

The famous claim is that directions in this space mean something. Take `king`,
subtract `man`, add `woman`, and you should land near `queen`.

| query | top 3 |
|---|---|
| `bigger - big + small` | **smaller**, larger, large |
| `walked - walk + run` | **ran**, running, runs |
| `paris - france + italy` | bologna, turin, **rome** |
| `king - man + woman` | throne, ermengarde, anjou |

Three work. The famous one does not.

The reason is about the data, not the method. This text is 17 million words of
Wikipedia. `king` appears mostly in lists of monarchs and succession prose. So its
vector leans towards monarchy and lineage, not towards male. Push it along the
gender direction and you land on the nearest female role in that same context.
`queen` comes fifteenth.

The gender direction is there. `father : mother :: son : daughter` lands first, and
so do three other family analogies. It is `king` in particular that is not mostly
about being a man here.

The same thing at the scale of the full run, on the 200 most frequent words rather
than thirty chosen ones, and through t-SNE rather than a flat projection:

![](figures/tsne_embeddings.png){fig-alt="t-SNE projection of the learned embedding space, the 200 most frequent words, with number words, nationalities and modal verbs each falling into their own region"}

## Conclusion

Week 1 starts with a feature engineering problem and ends with a way to solve it.

* Some features are already numbers
* Some categories can be decomposed into numbers by hand
* Free text can be neither, so you learn the numbers instead
* The training signal is free, because the text supplies its own labels

The second project of week 1 is the income model with a real dataset. Predict how
many upvotes a Hacker News post gets, from its title, its timestamp, its link and
its author. A number to predict, some easy features, and one free-text field.

These vectors fill that field in. [That post is
here](../2026-06-11-hacker-news-upvotes/).

![](figures/fig12-word2vec-bert-gpt.png){fig-alt="Three panels: word2vec predicting a masked word from an averaged window, BERT predicting two masked tokens from the full bidirectional sequence, and GPT predicting the next token causally"}

The window that makes this work is also what limits it. CBOW and skip-gram only
ever see a few words either side. Replace that fixed window with attention over the
whole sequence and you get BERT and GPT. That is where week 3 goes.

The full project, with the sweep and the deployed API, is
[on GitHub](https://github.com/RoshBeed/ai-residency/tree/main/services/word-embeddings).
